# آزمونِ میان‌بر روی مدلِ خودتان — نسخهٔ آمادهٔ اجرا روی GPU

این نوت‌بوک همان آزمایشِ مقاله را **دقیقاً روی داده و تعریف‌های خودِ شما** اجرا می‌کند: خطِ‌لولهٔ دادهٔ شما (پارسِ برچسب، استانداردسازیِ SMILES با RDKit، انگشت‌نگاریِ Morgan، تعریفِ `drug_pair`)، تقسیم‌بندیِ شما (`GroupKFold(5)` بر اساسِ `drug_pair`=LPO و `cell`=LCO) و هایپرپارامترهای شما (AdamW، lr=5e-5، wd=1e-2، pos_weight، grad-clip، ۱۰۰ epoch، patience=5، مانیتورِ AUC).

**ایده (طبقِ مقاله):** همه‌چیز ثابت می‌ماند و فقط **نمایشِ ورودی** عوض می‌شود. اگر حالتِ `ohe` (فقط شناسه) به‌اندازهٔ `full`/`drug_fp` خوب باشد، یعنی مدل میان‌بر می‌زند و از ویژگی‌ها یاد نمی‌گیرد.

**پنج حالت:** `full` (Morgan دو دارو + ویژگیِ ژنی از جمله‌های C2S)، `drug_fp` (فقط Morgan؛ همان baselineِ خودتان)، `ohe` (یک‌داغِ drug1+drug2+cell)، `shuffled` (کنترل) و `majority` (کفِ مقایسه).

**نحوهٔ استفاده:** فقط `DATA_PATH` را در سلولِ پیکربندی به فایلِ خودتان تنظیم کنید (بقیه آماده است). `BACKEND="torch"` روی GPU اجرا می‌شود. اگر خواستید سریع و بدونِ GPU چک کنید، `BACKEND="sklearn"` بگذارید.

> نکته: برای وفاداریِ کامل به روشِ مقاله، همهٔ حالت‌ها از **یک سرِ MLP یکسان** عبور می‌کنند و تنها بردارِ ورودی تغییر می‌کند (این همان کاری است که مقاله برای مدل‌های پیچیده انجام داد: انکودر را کنار می‌گذارد و ورودی را مستقیم به یک سرِ ساده می‌دهد). عددهای این آزمون را می‌توانید کنارِ عددهای مدلِ ترنسفورمرِ خودتان بگذارید.

### پیکربندی و ایمپورت‌ها — فقط `DATA_PATH` را عوض کنید

In [9]:
from scipy import stats
print("scipy OK")
print("run_paired defined:", 'run_paired' in dir())
print("MODES_P:", globals().get('MODES_P', 'NOT DEFINED'))

scipy OK
run_paired defined: True
MODES_P: ['full', 'drug_fp', 'ohe', 'ohe_cell', 'cell_mean', 'shuffled', 'shuffled_map', 'majority']


In [1]:
# ============================================================================
#  One-Hot Shortcut Test  —  adapted to YOUR model / data
#  ---------------------------------------------------------------------------
#  Same idea as Candir et al. 2026 (Bioinformatics): hold the model, the splits
#  and the hyper-parameters fixed, and change ONLY the input representation.
#  If one-hot IDs match your real features, the model is taking a shortcut
#  (learning identity, not chemistry/biology).
#
#  MODES compared (all through the SAME MLP head, exactly as the paper does):
#    - "full"      : Morgan(drug1) + Morgan(drug2) + gene features (from C2S sentences)
#    - "drug_fp"   : Morgan(drug1) + Morgan(drug2)                (your drug-only baseline)
#    - "ohe"       : one-hot(drug1) + one-hot(drug2) + one-hot(cell)   (identity only)
#    - "shuffled"  : drug_fp with feature rows permuted            (control)
#    - "majority"  : predict the training prior                    (floor baseline)
#
#  SPLITS (5-fold GroupKFold), matching your two notebook cells:
#    - group = "drug_pair"  -> Leave-Pair-Out  (LPO)   [your cell 1]
#    - group = "cell"       -> Leave-Cell-Out   (LCO)   [your cell 2]
#
#  Reuses YOUR data pipeline (label parsing, RDKit SMILES standardization,
#  Morgan FP, drug_pair) so the drug/cell identities are defined exactly as
#  in your model. Runs on your GPU. Requires: pandas numpy scikit-learn rdkit torch
# ============================================================================
import math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, f1_score, roc_auc_score)
from sklearn.feature_extraction.text import HashingVectorizer
warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG  (edit DATA_PATH to your file)
# ----------------------------------------------------------------------------
DATA_PATH   = r"C:\Users\afsha\Desktop\modell\finish_6\data\df_without_c2s_cols.csv\c2s27b_out_0_4396_in300_out300 (2).csv"
GROUPS      = ["drug_pair", "cell"]                       # LPO , LCO
MODES       = ["full", "drug_fp", "ohe", "ohe_cell", "cell_mean", "shuffled", "majority"]
N_SPLITS    = 5
SEED        = 42

# --- backends / fallbacks (delivered defaults) ---
BACKEND     = "torch"     # "torch" (GPU) or "sklearn" (CPU fallback / quick check)
USE_RDKIT   = True        # True: real Morgan FP ; False: hashed-SMILES fallback (no rdkit)

# --- model / training hyper-parameters (copied from your notebook) ---
FP_NBITS, FP_RADIUS = 1024, 2
GENE_HASH_DIM       = 512      # per-sentence block; full gene vec = 5*GENE_HASH_DIM
RANK_ALPHA          = 0.3      # matches the paper's initial rank weight
BATCH_SIZE          = 128
NUM_EPOCHS_FINAL    = 100
PATIENCE_FINAL      = 5
BEST_LR             = 5e-5
WEIGHT_DECAY        = 1e-2
GRAD_CLIP           = 1.0

gene_cols = ["cell_sentence", "c2s_treated_d1_sentence",
             "c2s_treated_d2_sentence", "c2s_treated_d12_direct"]
NULLISH = {"", "none", "nan", "null", "na", "n/a", "undefined", "missing"}

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)

# ----------------------------------------------------------------------------

### توابعِ کمکی (عیناً از نوت‌بوکِ خودتان) + استانداردسازیِ SMILES و انگشت‌نگاری

In [2]:
# helpers copied verbatim from your notebook (so data handling is identical)
# ----------------------------------------------------------------------------
def clean_text(x):
    if x is None: return ""
    s = str(x).strip()
    return "" if (not s) or (s.lower() in NULLISH) else s

def clean_smiles_string(x):
    if x is None: return None
    s = str(x).strip()
    return None if (not s) or (s.lower() in NULLISH) else s

def make_binary_label(df_):
    out = pd.Series([np.nan]*len(df_), index=df_.index, dtype="float")
    mapping = {"synergy":1,"antagonism":0,"1":1,"0":0,"true":1,"false":0,"yes":1,"no":0}
    if "label" in df_.columns:
        num = pd.to_numeric(df_["label"], errors="coerce")
        mapped = df_["label"].astype(str).str.strip().str.lower().map(mapping)
        out = num.fillna(mapped)
    if out.isna().any() and "classification" in df_.columns:
        out = out.fillna(df_["classification"].astype(str).str.strip().str.lower().map(mapping))
    return out

def safe_auc(y_true, y_prob):
    try:
        return roc_auc_score(y_true, y_prob)
    except Exception:
        return 0.5

def threshold_sweep_0_to_100(y_true, y_prob):
    best_f1, best_thr = -1.0, 0.5
    for t in range(101):
        thr = t/100.0
        f1 = f1_score(y_true, (y_prob >= thr).astype(int), zero_division=0)
        if f1 > best_f1: best_f1, best_thr = f1, thr
    return best_thr, best_f1

# --- SMILES standardization + Morgan FP (rdkit, with a no-rdkit fallback) ---
if USE_RDKIT:
    try:
        from rdkit import Chem, RDLogger
        from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
        RDLogger.DisableLog("rdApp.*")
        _HAVE_RDKIT = True
    except Exception:
        _HAVE_RDKIT = False
        print("[warn] rdkit not found -> using hashed-SMILES fingerprint fallback")
else:
    _HAVE_RDKIT = False

def standardize_smiles(s):
    s = clean_smiles_string(s)
    if not s: return None
    if not _HAVE_RDKIT:
        return s                      # fallback: use raw string as identity
    mol = Chem.MolFromSmiles(s)
    if mol is None: return None
    try:
        from rdkit.Chem.MolStandardize import rdMolStandardize
        mol = rdMolStandardize.LargestFragmentChooser().choose(mol)
        mol = rdMolStandardize.Uncharger().uncharge(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except Exception:
        return None

def smiles_to_fingerprint(smiles):
    if not _HAVE_RDKIT:               # deterministic hashed fallback
        rng = np.random.default_rng(abs(hash(smiles)) % (2**32))
        v = np.zeros(FP_NBITS, dtype=np.float32); v[rng.integers(0, FP_NBITS, 32)] = 1.0
        return v
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(FP_NBITS, dtype=np.float32)
    gen = GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_NBITS)
    return np.array(gen.GetFingerprint(mol), dtype=np.float32)

# ----------------------------------------------------------------------------

### بارگذاری و آماده‌سازیِ داده (مطابقِ خطِ‌لولهٔ شما)

In [3]:
# 1) load + prepare data  (mirrors your notebook)
# ----------------------------------------------------------------------------
def load_data(path):
    df_path = Path(path)
    if not df_path.exists():
        for c in [Path.cwd()/"df_with_c2s.csv.gz", Path.cwd()/"c2s27b_out_0_4396_in300_out300 (2).csv"]:
            if c.exists(): df_path = c; break
    if not df_path.exists():
        raise FileNotFoundError(f"data file not found: {path}")
    is_gz = str(df_path).endswith(".gz")
    print("Using data file:", df_path)
    cols = pd.read_csv(df_path, compression="gzip" if is_gz else None, nrows=0).columns.tolist()
    usecols = [c for c in (["cell","drug1_smiles","drug2_smiles"]+gene_cols+["label","classification"]) if c in cols]
    df = pd.read_csv(df_path, compression="gzip" if is_gz else None, usecols=usecols)

    df["label"] = make_binary_label(df)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    df["label"] = df["label"].astype(float).round().astype(int).clip(0, 1)
    df["cell"] = df["cell"].astype(str)
    for c in gene_cols:
        if c in df.columns: df[c] = df[c].apply(clean_text)

    print("Standardizing SMILES ...")
    df["drug1_smiles"] = df["drug1_smiles"].apply(standardize_smiles)
    df["drug2_smiles"] = df["drug2_smiles"].apply(standardize_smiles)
    df = df[df["drug1_smiles"].notna() & df["drug2_smiles"].notna()].reset_index(drop=True)

    print("Computing Morgan fingerprints ...")
    df["fp1"] = df["drug1_smiles"].apply(smiles_to_fingerprint)
    df["fp2"] = df["drug2_smiles"].apply(smiles_to_fingerprint)
    df["drug_pair"] = df.apply(lambda r: "||".join(sorted([r["drug1_smiles"], r["drug2_smiles"]])), axis=1)
    return df

# ----------------------------------------------------------------------------

### ساختِ ماتریس‌های ویژگی (یک‌بار روی کلِ داده تا ستون‌ها بینِ فولدها هم‌تراز بمانند)

In [4]:
# 2) feature matrices  — CORRECTED
# ----------------------------------------------------------------------------
#  Fixes vs. the original:
#   (a) gene sentences are encoded SEPARATELY and with RANK WEIGHTS w_i=1/(1+i)^a,
#       then combined into the paper's differential tokens
#       {z_base, e12, e_sum, e_abs, e_mul}.  The original merged all four
#       sentences into one binary bag-of-genes, which destroys both the rank
#       encoder and the differential encoding (the paper's contributions 2 & 3).
#   (b) one-hot drugs use ONE shared vocabulary over the union and are summed
#       -> order-invariant multi-hot, consistent with the sorted `drug_pair`
#       grouping and with the paper's symmetric drug tokens.
#   (c) hard failure (not silent fallback) if the C2S sentence columns are
#       absent or empty, which would make `full` identical to `drug_fp`.
#   (d) stable hashing (zlib.crc32) so runs are reproducible across processes.
# ----------------------------------------------------------------------------
import zlib

def _h(gene, dim):
    return zlib.crc32(gene.encode("utf-8")) % dim

def _rank_encode(series, dim, alpha):
    """Rank-weighted multi-hot: the gene at rank i contributes 1/(1+i)**alpha."""
    M = np.zeros((len(series), dim), dtype=np.float32)
    for r, s in enumerate(series.values):
        toks = str(s).split()
        for i, g in enumerate(toks):
            M[r, _h(g, dim)] += 1.0 / (1.0 + i) ** alpha
    return M

def build_features(df):
    F = {}
    F["fp1"] = np.stack(df["fp1"].values).astype(np.float32)
    F["fp2"] = np.stack(df["fp2"].values).astype(np.float32)

    # ---- (b) order-invariant one-hot over the union of drugs -----------------
    drugs = sorted(set(df["drug1_smiles"]) | set(df["drug2_smiles"]))
    didx  = {s: i for i, s in enumerate(drugs)}
    ohd = np.zeros((len(df), len(drugs)), dtype=np.float32)
    for r, (a, b) in enumerate(zip(df["drug1_smiles"], df["drug2_smiles"])):
        ohd[r, didx[a]] += 1.0
        ohd[r, didx[b]] += 1.0
    ohc = pd.get_dummies(df["cell"], prefix="c").astype(np.float32).to_numpy()
    F["ohe"]      = np.concatenate([ohd, ohc], axis=1)   # drug IDs + cell ID
    F["ohe_cell"] = ohc                                  # cell ID alone
    print(f"one-hot: {len(drugs)} unique drugs (union), {ohc.shape[1]} cell-lines")

    # ---- (c) fail loudly if the gene sentences are not really there ----------
    missing = [c for c in gene_cols if c not in df.columns]
    if missing:
        raise KeyError(f"gene columns missing from the CSV: {missing}. "
                       "Without them 'full' would silently equal 'drug_fp'.")
    counts = {c: int((df[c].astype(str).str.strip().str.len() > 0).sum()) for c in gene_cols}
    print("non-empty sentences per column:", counts)
    for c, n in counts.items():
        if n < 0.5 * len(df):
            raise ValueError(f"column '{c}' is empty in {len(df)-n}/{len(df)} rows -> "
                             "'full' would collapse onto 'drug_fp'. Check DATA_PATH.")

    # ---- (a) rank-weighted, per-sentence, then differential tokens -----------
    D = GENE_HASH_DIM
    Zb  = _rank_encode(df["cell_sentence"],            D, RANK_ALPHA)
    Z1  = _rank_encode(df["c2s_treated_d1_sentence"],  D, RANK_ALPHA)
    Z2  = _rank_encode(df["c2s_treated_d2_sentence"],  D, RANK_ALPHA)
    Z12 = _rank_encode(df["c2s_treated_d12_direct"],   D, RANK_ALPHA)

    e12  = Z12 - Zb                      # observed combined shift
    esum = (Z1 - Zb) + (Z2 - Zb)         # additive expectation
    eabs = np.abs(Z1 - Z2)
    emul = Z1 * Z2
    F["gene"] = np.concatenate([Zb, e12, esum, eabs, emul], axis=1)

    # sanity: the differential signal must not be degenerate
    frac = float(np.mean(np.abs(e12 - esum) > 1e-6))
    print(f"gene block: {F['gene'].shape[1]} dims | rows where e12 != e_sum: {frac:.1%}")
    if frac < 0.01:
        raise ValueError("e12 == e_sum almost everywhere -> C2S returned (near) identical "
                         "sentences for the combination and the singles. Inspect the data.")
    return F

def assemble(F, mode, seed=0):
    if mode == "drug_fp":
        return np.concatenate([F["fp1"], F["fp2"]], axis=1)
    if mode == "full":
        return np.concatenate([F["fp1"], F["fp2"], F["gene"]], axis=1)
    if mode == "ohe":
        return F["ohe"]
    if mode == "ohe_cell":                      # drug chemistry + cell IDENTITY only
        return np.concatenate([F["fp1"], F["fp2"], F["ohe_cell"]], axis=1)
    if mode == "shuffled":
        X = np.concatenate([F["fp1"], F["fp2"]], axis=1).copy()
        rng = np.random.default_rng(seed); d = F["fp1"].shape[1]
        X[:, :d] = X[rng.permutation(X.shape[0]), :d]
        X[:, d:] = X[rng.permutation(X.shape[0]), d:]
        return X
    raise ValueError(mode)


### مدل: یک MLP مشترک که بُعدِ ورودی‌اش با هر حالت تطبیق می‌یابد + آموزش با early-stopping

In [5]:
# 3) model  (a single MLP head; input dim adapts to the mode)  + training
# ----------------------------------------------------------------------------
def train_predict(X_tr, y_tr, X_va, y_va, X_te, pos_weight, seed=0):
    if BACKEND == "sklearn":
        from sklearn.neural_network import MLPClassifier
        clf = MLPClassifier(hidden_layer_sizes=(512, 128), activation="relu",
                            alpha=WEIGHT_DECAY, max_iter=NUM_EPOCHS_FINAL,
                            early_stopping=True, n_iter_no_change=PATIENCE_FINAL,
                            random_state=seed)
        clf.fit(X_tr, y_tr)
        return clf.predict_proba(X_va)[:, 1], clf.predict_proba(X_te)[:, 1]

    # ---- torch backend (default, uses GPU) ----
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    torch.manual_seed(seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    class MLP(nn.Module):
        def __init__(self, d_in):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(d_in, 512), nn.GELU(), nn.Dropout(0.30),
                nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.30),
                nn.Linear(128, 1))
        def forward(self, x): return self.net(x).squeeze(-1)

    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.float32))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

    tr_ld, va_ld = loader(X_tr, y_tr, True), loader(X_va, y_va, False)
    model = MLP(X_tr.shape[1]).to(dev)
    crit  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=dev))
    opt   = torch.optim.AdamW(model.parameters(), lr=BEST_LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)

    best_auc, best_state, wait = -1.0, None, 0
    for epoch in range(NUM_EPOCHS_FINAL):
        model.train()
        for xb, yb in tr_ld:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        # validation AUC (early-stopping monitor, exactly as in your notebook)
        model.eval(); probs, trues = [], []
        with torch.no_grad():
            for xb, yb in va_ld:
                probs.append(torch.sigmoid(model(xb.to(dev))).cpu().numpy()); trues.append(yb.numpy())
        va_auc = safe_auc(np.concatenate(trues), np.concatenate(probs))
        sched.step(va_auc)
        if va_auc > best_auc + 1e-5:
            best_auc, wait = va_auc, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE_FINAL: break
    if best_state is not None: model.load_state_dict(best_state)

    def infer(X):
        model.eval(); out = []
        with torch.no_grad():
            ld = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)),
                            batch_size=BATCH_SIZE, shuffle=False)
            for (xb,) in ld:
                out.append(torch.sigmoid(model(xb.to(dev))).cpu().numpy())
        return np.concatenate(out)
    # returns (validation probs, test probs) so the threshold is picked on VAL
    return infer(X_va), infer(X_te)

# ----------------------------------------------------------------------------

### اجرای اعتبارسنجیِ متقاطع و گزارشِ معیارها

In [6]:
# 4) cross-validated shortcut test — CORRECTED
# ----------------------------------------------------------------------------
#  Fixes: the decision threshold is now selected on the VALIDATION fold and
#  applied to the test fold (the original swept it on the test labels, which
#  inflates F1/ACC/BalAcc and does not match the paper's protocol).
#  Adds `cell_mean`: predict the training synergy rate of the test row's
#  cell-line — the strongest identity-only baseline in an LPO split.
# ----------------------------------------------------------------------------
def fold_metrics(y_true, y_prob, thr):
    yp = (y_prob >= thr).astype(int)
    try:    prauc = average_precision_score(y_true, y_prob)
    except Exception: prauc = np.nan
    return dict(AUC=safe_auc(y_true, y_prob), F1=f1_score(y_true, yp, zero_division=0),
                ACC=accuracy_score(y_true, yp), PRAUC=prauc,
                BALACC=balanced_accuracy_score(y_true, yp))

def run(df, F, group, modes):
    gkf = GroupKFold(n_splits=N_SPLITS)
    y   = df["label"].to_numpy()
    cells_arr = df["cell"].to_numpy()
    rows = []
    for mode in modes:
        per = []
        for fold, (tr, te) in enumerate(gkf.split(df, y, groups=df[group]), 1):
            rng   = np.random.default_rng(SEED + fold)
            g_tr  = df[group].to_numpy()[tr]
            uniq  = np.array(sorted(set(g_tr))); rng.shuffle(uniq)
            va_g  = set(uniq[:max(1, int(0.15 * len(uniq)))])
            va_m  = np.isin(g_tr, list(va_g))
            tr_idx, va_idx = tr[~va_m], tr[va_m]
            if len(va_idx) == 0: tr_idx, va_idx = tr[:-1], tr[-1:]

            if mode == "majority":
                p = y[tr_idx].mean()
                pv = np.full(len(va_idx), p); pt = np.full(len(te), p)
            elif mode == "cell_mean":
                s = pd.Series(y[tr_idx]).groupby(cells_arr[tr_idx]).mean()
                g = y[tr_idx].mean()
                pv = np.array([s.get(c, g) for c in cells_arr[va_idx]], dtype=float)
                pt = np.array([s.get(c, g) for c in cells_arr[te]],     dtype=float)
            else:
                X   = assemble(F, mode, seed=SEED + fold)
                pos = max(int(y[tr_idx].sum()), 1); neg = len(tr_idx) - pos
                pv, pt = train_predict(X[tr_idx], y[tr_idx].astype(np.float32),
                                       X[va_idx], y[va_idx].astype(np.float32),
                                       X[te], pos_weight=neg / pos, seed=SEED + fold)
            thr, _ = threshold_sweep_0_to_100(y[va_idx], pv)   # threshold from VAL
            per.append(fold_metrics(y[te], pt, thr))
        agg = {k: (np.nanmean([d[k] for d in per]), np.nanstd([d[k] for d in per])) for k in per[0]}
        rows.append((mode, agg))
        print(f"  [{group:9s}] {mode:10s} AUC={agg['AUC'][0]:.3f}")
    return rows

def print_table(group, rows):
    label = {"drug_pair": "LPO (leave-pair-out)", "cell": "LCO (leave-cell-out)"}[group]
    print(f"\n{'='*82}\nSPLIT = {label}   [group = '{group}']   {N_SPLITS}-fold GroupKFold\n{'='*82}")
    print(f"{'mode':11s} {'AUC':>14s} {'F1':>14s} {'ACC':>13s} {'PR-AUC':>13s} {'BalAcc':>13s}")
    for mode, agg in rows:
        def c(k): return f"{agg[k][0]:.3f}\u00b1{agg[k][1]:.3f}"
        print(f"{mode:11s} {c('AUC'):>14s} {c('F1'):>14s} {c('ACC'):>13s} {c('PRAUC'):>13s} {c('BALACC'):>13s}")


### اجرا — این سلول همه‌چیز را اجرا و جدولِ مقایسه را چاپ می‌کند

In [7]:
seed_everything(SEED)
df = load_data(DATA_PATH)
print(f"\nrows: {len(df)} | drug pairs: {df['drug_pair'].nunique()} | cells: {df['cell'].nunique()} | pos-rate: {df['label'].mean():.3f}")
F = build_features(df)
for g in GROUPS:
    if df[g].nunique() < N_SPLITS:
        print(f"[skip] group '{g}' has < {N_SPLITS} unique values"); continue
    print_table(g, run(df, F, g, MODES))
print("\nRead the LPO row 'ohe' vs 'full'/'drug_fp': if they are close, the model is taking the shortcut.")

Using data file: C:\Users\afsha\Desktop\modell\finish_6\data\df_without_c2s_cols.csv\c2s27b_out_0_4396_in300_out300 (2).csv
Standardizing SMILES ...
Computing Morgan fingerprints ...

rows: 3885 | drug pairs: 3841 | cells: 52 | pos-rate: 0.319
one-hot: 106 unique drugs (union), 52 cell-lines
non-empty sentences per column: {'cell_sentence': 3885, 'c2s_treated_d1_sentence': 3885, 'c2s_treated_d2_sentence': 3885, 'c2s_treated_d12_direct': 3885}
gene block: 2560 dims | rows where e12 != e_sum: 53.7%
  [drug_pair] full       AUC=0.765
  [drug_pair] drug_fp    AUC=0.714
  [drug_pair] ohe        AUC=0.749
  [drug_pair] ohe_cell   AUC=0.733
  [drug_pair] cell_mean  AUC=0.682
  [drug_pair] shuffled   AUC=0.485
  [drug_pair] majority   AUC=0.500

SPLIT = LPO (leave-pair-out)   [group = 'drug_pair']   5-fold GroupKFold
mode                   AUC             F1           ACC        PR-AUC        BalAcc
full           0.765±0.016    0.577±0.012   0.721±0.030   0.667±0.034   0.688±0.011
drug_fp    

### چطور نتیجه را بخوانیم

سه مقایسه مهم است، به همین ترتیب:

**۱. `full` در برابرِ `ohe_cell` — تزِ واقعیِ مقاله.**
`ohe_cell` همان شیمیِ دارو است به‌علاوهٔ *هویتِ* ردهٔ سلولی، بدونِ هیچ زیست‌شناسی.
اگر `full` از آن جلو بزند، نشان داده‌ای پرتوربیشنِ تولیدشده چیزی **فراتر از هویتِ سلول**
حمل می‌کند — دقیقاً همان چیزی که Çandır می‌گوید تا امروز هیچ مدلی نشان نداده.
اگر برابر درآیند، ΔAUC = ۰٫۰۲۷ مقاله‌ات اثرِ هویت است نه زیست‌شناسی.

**۲. `full` در برابرِ `cell_mean`.** ساده‌ترین پیش‌بینِ هویتی: نرخِ هم‌افزاییِ آموزشیِ
همان ردهٔ سلولی. با A2058 روی ۰٫۷۷ در برابرِ ۰٫۳۲ کلی، این در LPO احتمالاً قوی است.
هر مدلی که از این جلو نزند، عملاً چیزی یاد نگرفته.

**۳. `ohe` در برابرِ `full`/`drug_fp` — آزمونِ میان‌بُرِ Çandır.**
اگر یک‌داغ هم‌رده باشد، مدل دارد هویت یاد می‌گیرد نه شیمی/زیست.

**دو کنترل:** `shuffled` توزیعِ ویژگی را نگه می‌دارد ولی پیوندش با دارو را می‌شکند —
اگر `drug_fp ≈ shuffled` یعنی محتوای انگشت‌نگاری بی‌اثر است. `majority` کفِ مطلق است.

---

**در LCO حواست باشد:** رده‌های سلولیِ تست دیده‌نشده‌اند، پس ستون‌های یک‌داغِ آن‌ها در
آموزش تماماً صفرند و بلوکِ سلول هیچ سهمی ندارد. یعنی `ohe` در LCO فقط هویتِ **دارو** را
می‌سنجد و `ohe_cell` عملاً به `drug_fp` فرو می‌کاهد. این طبیعی است، ولی نتیجه را
باید همین‌طور خواند. جای واقعیِ برتریِ روشِ تو هم همین‌جاست: C2S برای ردهٔ دیده‌نشده
هم جمله تولید می‌کند، کاری که یک‌داغ اصلاً نمی‌تواند.

**نکتهٔ آماری:** `GroupKFold` تصادفی نیست، پس تغییرِ `SEED` اسپلیت را عوض نمی‌کند —
فقط ولیدیشن و مقداردهیِ اولیه را. برای سنجشِ شکنندگیِ p = ۰٫۰۵۰۱ باید `GroupShuffleSplit`
یا جابه‌جاییِ تصادفیِ گروه‌ها را جایگزین کنی.


---
## آزمونِ زوجی — این سلول را بعد از اجرای سلولِ بالا اجرا کنید

سلولِ قبلی فقط میانگین و انحرافِ معیار می‌دهد. چون هر پنج فولد بینِ حالت‌ها **مشترک**‌اند،
مقایسهٔ درست، مقایسهٔ **زوجی** است: تفاوتِ هر فولد جداگانه حساب می‌شود و بعد آزمون
روی همان پنج تفاوت اجرا می‌شود. این کار توانِ آماریِ به‌مراتب بیشتری دارد.

این سلول سه چیز اضافه می‌کند:

1. مقادیرِ **هر فولد** نگه داشته می‌شود (سلولِ قبلی فقط میانگین می‌گرفت).
2. آزمونِ **t زوجی** به‌همراه اندازهٔ اثرِ `dz` برای پنج جفتِ مقایسه.
3. کنترلِ `shuffled_map` که وفادار به روشِ Çandır است: نگاشتِ دارو→انگشت‌نگاری
   به‌طور **سازگار** جابه‌جا می‌شود (هر دارو انگشت‌نگاریِ داروی دیگری می‌گیرد)، پس
   هویت از طریقِ ویژگی‌ها قابلِ بازیابی می‌ماند. `shuffled` قبلی سطرها را مستقل
   جابه‌جا می‌کرد که کنترلِ سخت‌گیرانه‌تری است. هر دو اجرا می‌شوند.

`scipy` لازم است. اجرای این سلول تقریباً دو برابرِ سلولِ قبلی طول می‌کشد.


In [8]:
# =============================================================================
#  PATCH — paste as a new cell AFTER cell 12, then re-run the final cell.
#  Adds: (1) per-fold values kept, (2) paired t-tests between modes,
#        (3) a Candir-faithful shuffle control (`shuffled_map`).
# =============================================================================
from scipy import stats

# --- (3) Candir-style shuffle: permute the drug -> fingerprint LOOKUP, so each
#         drug consistently receives some *other* drug's fingerprint. Identity
#         stays recoverable; only the chemistry is wrong. The original
#         `shuffled` permutes rows independently, which is a harsher control.
if not globals().get('_PATCHED_ASSEMBLE'):      # safe to re-run this cell
    _orig_assemble = assemble
    _PATCHED_ASSEMBLE = True

def assemble(F, mode, seed=0):
    if mode == "shuffled_map":
        rng = np.random.default_rng(seed)
        drugs = sorted(set(_DF["drug1_smiles"]) | set(_DF["drug2_smiles"]))
        perm = {d: drugs[i] for d, i in zip(drugs, rng.permutation(len(drugs)))}
        fp = {}
        for d, f in zip(_DF["drug1_smiles"], _DF["fp1"]): fp.setdefault(d, np.asarray(f, np.float32))
        for d, f in zip(_DF["drug2_smiles"], _DF["fp2"]): fp.setdefault(d, np.asarray(f, np.float32))
        a = np.stack([fp[perm[d]] for d in _DF["drug1_smiles"]]).astype(np.float32)
        b = np.stack([fp[perm[d]] for d in _DF["drug2_smiles"]]).astype(np.float32)
        return np.concatenate([a, b], axis=1)
    return _orig_assemble(F, mode, seed=seed)


def run_paired(df, F, group, modes):
    """Same as run(), but keeps every fold's metrics so modes can be compared pairwise."""
    global _DF
    _DF = df
    gkf = GroupKFold(n_splits=N_SPLITS)
    y, cells_arr = df["label"].to_numpy(), df["cell"].to_numpy()
    folds = list(gkf.split(df, y, groups=df[group]))
    per_mode = {}

    for mode in modes:
        vals = []
        for fold, (tr, te) in enumerate(folds, 1):
            rng  = np.random.default_rng(SEED + fold)
            g_tr = df[group].to_numpy()[tr]
            uniq = np.array(sorted(set(g_tr))); rng.shuffle(uniq)
            va_g = set(uniq[:max(1, int(0.15 * len(uniq)))])
            va_m = np.isin(g_tr, list(va_g))
            tr_idx, va_idx = tr[~va_m], tr[va_m]
            if len(va_idx) == 0: tr_idx, va_idx = tr[:-1], tr[-1:]

            if mode == "majority":
                p = y[tr_idx].mean()
                pv, pt = np.full(len(va_idx), p), np.full(len(te), p)
            elif mode == "cell_mean":
                s = pd.Series(y[tr_idx]).groupby(cells_arr[tr_idx]).mean()
                g = y[tr_idx].mean()
                pv = np.array([s.get(c, g) for c in cells_arr[va_idx]], float)
                pt = np.array([s.get(c, g) for c in cells_arr[te]],     float)
            else:
                X = assemble(F, mode, seed=SEED + fold)
                pos = max(int(y[tr_idx].sum()), 1); neg = len(tr_idx) - pos
                pv, pt = train_predict(X[tr_idx], y[tr_idx].astype(np.float32),
                                       X[va_idx], y[va_idx].astype(np.float32),
                                       X[te], pos_weight=neg / pos, seed=SEED + fold)
            thr, _ = threshold_sweep_0_to_100(y[va_idx], pv)
            vals.append(fold_metrics(y[te], pt, thr))
        per_mode[mode] = vals
        print(f"  [{group:9s}] {mode:13s} AUC={np.mean([v['AUC'] for v in vals]):.3f}")
    return per_mode


def paired_report(per_mode, group, pairs, metrics=("AUC", "PRAUC")):
    label = {"drug_pair": "LPO (leave-pair-out)", "cell": "LCO (leave-cell-out)"}[group]
    print(f"\n{'='*78}\nPAIRED COMPARISONS — {label}\n{'='*78}")
    for a, b in pairs:
        if a not in per_mode or b not in per_mode: continue
        print(f"\n{a}  vs  {b}")
        for m in metrics:
            va = np.array([v[m] for v in per_mode[a]])
            vb = np.array([v[m] for v in per_mode[b]])
            d  = va - vb
            t, p = stats.ttest_rel(va, vb)
            # Cohen's dz for paired samples
            dz = d.mean() / d.std(ddof=1) if d.std(ddof=1) > 0 else np.nan
            stars = "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"
            print(f"   {m:6s} {va.mean():.4f} vs {vb.mean():.4f} | "
                  f"delta={d.mean():+.4f} +/- {d.std(ddof=1):.4f} | "
                  f"t={t:+.3f}  p={p:.4f} {stars}  dz={dz:+.2f}")
            print(f"          per-fold deltas: {np.array2string(d, precision=4)}")


# ----------------------------------------------------------------------------- run
MODES_P = ["full", "drug_fp", "ohe", "ohe_cell", "cell_mean",
           "shuffled", "shuffled_map", "majority"]   # both shuffle controls
PAIRS = [("full", "ohe_cell"),   # <-- the paper's real thesis
         ("full", "ohe"),        # <-- beats pure identity?
         ("full", "drug_fp"),    # <-- the paper's current claim
         ("ohe",  "drug_fp"),    # <-- is the shortcut present?
         ("drug_fp", "shuffled_map")]  # <-- is the chemistry used?

for g in GROUPS:
    if df[g].nunique() < N_SPLITS:
        print(f"[skip] group '{g}'"); continue
    pm = run_paired(df, F, g, MODES_P)
    paired_report(pm, g, PAIRS)


  [drug_pair] full          AUC=0.765
  [drug_pair] drug_fp       AUC=0.714
  [drug_pair] ohe           AUC=0.749
  [drug_pair] ohe_cell      AUC=0.733
  [drug_pair] cell_mean     AUC=0.682
  [drug_pair] shuffled      AUC=0.485
  [drug_pair] shuffled_map  AUC=0.695
  [drug_pair] majority      AUC=0.500

PAIRED COMPARISONS — LPO (leave-pair-out)

full  vs  ohe_cell
   AUC    0.7646 vs 0.7335 | delta=+0.0311 +/- 0.0151 | t=+4.590  p=0.0101 *  dz=+2.05
          per-fold deltas: [0.0289 0.0436 0.0354 0.006  0.0416]
   PRAUC  0.6671 vs 0.6107 | delta=+0.0565 +/- 0.0317 | t=+3.983  p=0.0164 *  dz=+1.78
          per-fold deltas: [0.0406 0.0585 0.0476 0.0264 0.1093]

full  vs  ohe
   AUC    0.7646 vs 0.7493 | delta=+0.0153 +/- 0.0042 | t=+8.218  p=0.0012 **  dz=+3.68
          per-fold deltas: [0.0091 0.0171 0.0182 0.0129 0.0189]
   PRAUC  0.6671 vs 0.6521 | delta=+0.0150 +/- 0.0087 | t=+3.862  p=0.0181 *  dz=+1.73
          per-fold deltas: [0.0057 0.0225 0.0219 0.0054 0.0197]

full  vs  dr

In [10]:
from scipy import stats
import numpy as np

y, gkf = df["label"].to_numpy(), GroupKFold(n_splits=N_SPLITS)
res = {}
for mode in ["full", "ohe_cell"]:
    per = []
    for fold, (tr, te) in enumerate(gkf.split(df, y, groups=df["drug_pair"]), 1):
        rng  = np.random.default_rng(SEED + fold)
        g_tr = df["drug_pair"].to_numpy()[tr]
        uniq = np.array(sorted(set(g_tr))); rng.shuffle(uniq)
        va_m = np.isin(g_tr, list(set(uniq[:max(1, int(0.15 * len(uniq)))])))
        tr_idx, va_idx = tr[~va_m], tr[va_m]
        X = _orig_assemble(F, mode, seed=SEED + fold)
        pos = max(int(y[tr_idx].sum()), 1); neg = len(tr_idx) - pos
        pv, pt = train_predict(X[tr_idx], y[tr_idx].astype(np.float32),
                               X[va_idx], y[va_idx].astype(np.float32),
                               X[te], pos_weight=neg / pos, seed=SEED + fold)
        thr, _ = threshold_sweep_0_to_100(y[va_idx], pv)
        per.append(fold_metrics(y[te], pt, thr))
    res[mode] = per
    print(f"{mode:9s} AUC={np.mean([v['AUC'] for v in per]):.4f}")

print("\n--- PAIRED: full vs ohe_cell (LPO) ---")
for m in ["AUC", "PRAUC"]:
    a = np.array([v[m] for v in res["full"]])
    b = np.array([v[m] for v in res["ohe_cell"]])
    d = a - b
    t, p = stats.ttest_rel(a, b)
    print(f"{m:6s} {a.mean():.4f} vs {b.mean():.4f} | delta={d.mean():+.4f} "
          f"| t={t:+.3f} p={p:.4f} | per-fold: {np.round(d,4)}")

full      AUC=0.7646
ohe_cell  AUC=0.7335

--- PAIRED: full vs ohe_cell (LPO) ---
AUC    0.7646 vs 0.7335 | delta=+0.0311 | t=+4.590 p=0.0101 | per-fold: [0.0289 0.0436 0.0354 0.006  0.0416]
PRAUC  0.6671 vs 0.6107 | delta=+0.0565 | t=+3.983 p=0.0164 | per-fold: [0.0406 0.0585 0.0476 0.0264 0.1093]


### کدام عدد به مقاله می‌رود

**فقط بلوکِ `full  vs  ohe_cell`.** این تزِ واقعیِ مقاله است: هر دو طرف شیمیِ داروی
یکسان دارند و تنها تفاوتشان این است که ردهٔ سلولی با جملهٔ C2S توصیف شده یا با یک شناسه.

در بخش ۴٫۵ مقاله این جمله هست:

> The gap of 0.032 in ROC-AUC is about twice the fold-to-fold standard deviation of either arm

بعد از اجرا، جایگزینش کن با نتیجهٔ زوجی، مثلاً:

> The gap of 0.032 in ROC-AUC is significant under a paired t-test across the five folds
> (t = …, p = …).

**از بلوکِ `full vs ohe` استفاده نکن.** فاصله‌اش ۰٫۰۱۶ است که در حدِ یک انحرافِ معیارِ
فولد است، و در متنِ مقاله صریح نوشته شده که این را ادعا نمی‌کنیم. حتی اگر اتفاقاً
معنادار دربیاید، با پنج فولد قابلِ اتکا نیست.

بلوک‌های `ohe vs drug_fp` و `drug_fp vs shuffled_map` هم اطلاعاتی‌اند: اولی نشان می‌دهد
میان‌بُر چقدر قوی است، دومی نشان می‌دهد محتوای شیمیایی واقعاً استفاده می‌شود.
اگر عددِ `shuffled_map` خیلی با `shuffled` فرق داشت، ارزشش را دارد هر دو را در جدولِ ۵
گزارش کنی.
